In [1]:
# fix imports
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

In [2]:
from src.eval import HarmBenchEvaluator, LlamaEvaluator, TemplateEvaluator, BeaverEvaluator
from gserve.configs import ServeConfig, LLMConfig
import os


evaluators = [
    # HarmBenchEvaluator(
    #     serve_config=ServeConfig(gpu_ids=[1], startup_timeout=5 * 60, client_timeout=60),
    #     use_context=False,
    # ),
    # LlamaEvaluator(
    #     serve_config=ServeConfig(gpu_ids=[1], startup_timeout=5 * 60, client_timeout=60),
    # ),
    BeaverEvaluator(
        device_map="cpu",
    ),
    TemplateEvaluator(),
]

INFO 09-26 16:04:48 [__init__.py:248] No platform detected, vLLM is running on UnspecifiedPlatform


Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

In [ ]:
from vllm import LLM, SamplingParams

model = LLM(
    model="meta-llama/Llama-2-7b-chat-hf",
    # tensor_parallel_size=1,
    dtype="bfloat16",
)

sampling_params = SamplingParams(
    temperature=0,
    max_tokens=100,
)

In [3]:
import pandas as pd
from src.data import DF_Batcher

data = pd.read_csv("eval_data.csv")
ds_eval = data.copy().iloc[:10]

# make all entries in response column max length 10
ds_eval["response"] = ds_eval["response"].apply(lambda x: x[:10])

dl_eval = DF_Batcher(ds_eval, batch_size=200, shuffle=False)

In [ ]:
data

In [ ]:
# make all 

In [ ]:
from tqdm.auto import tqdm

all_outputs = []
for batch in tqdm(dl_eval):
    convos = [[{"role": "user", "content": prompt}] for prompt in batch["prompt"]]
    outputs = model.chat(convos, sampling_params=sampling_params)
    all_outputs.extend([output.outputs[0].text for output in outputs])

dl_eval.set_column("response", all_outputs)

In [ ]:
ds_eval

In [4]:
eval_results = {}

for evaluator in evaluators:
    print(f"Running evaluator: {evaluator.name}")
    results = evaluator.evaluate(dl_eval)
    eval_results.update(results)
    print(f"Results: {results}")

Running evaluator: Beaver


Evaluating Beaver:   0%|          | 0/1 [00:00<?, ?it/s]

Results: {'Beaver/Raw': 4.065625, 'Beaver/Thresh@2.5': 0.9, 'Beaver/Thresh@5': 0.3, 'Beaver/Thresh@7.5': 0.0, 'Beaver/Thresh@10': 0.0}
Running evaluator: Template
Results: {'Template': 1.0}


In [ ]:
ds_eval

In [ ]:
# print prompts and outputs

for i, row in dl_eval.df.iterrows():
    print(f"Prompt: {row['prompt']}")
    print(f"Response: {row['response']}")
    print("-" * 80)